# Flood susceptibility model: explore it yourself

## 1. What this is

This notebook lets you look inside a small machine-learning model for Barcelona streets. You do not need to know any code. Just run the cells from top to bottom (Runtime > Run all in Colab).

**What the model does.** It learns which street segments flood in a *physics simulation*, then answers what-if questions such as "how would the streets look under 75 mm of rain with weaker drains?". It shows **susceptibility under a what-if scenario**. It does not say what the weather will do, and it does not say what real floods will do.

**What it was trained on.** Only 2 simulated storms: a light one on 2 Oct (15.5 mm of rain) and a heavy one on 3 Oct (143.4 mm). Every "flooded" label comes from the simulation, not from real reports. It is a prototype and trust is **low**. The cell below shows the trust level and the reasons, taken from `metrics.json`.

**How to read the numbers.** AUC is a score from 0.5 (no better than a coin flip) to 1.0 (perfect ranking). Accuracy can fool you, so we always show it next to a "just guess dry" baseline.

In [ ]:
# Settings and imports. Run this cell first. Nothing here needs editing, except DATA_FOLDER if you named your Drive folder differently.
import importlib, subprocess, sys, os, json, math, shutil, warnings
from pathlib import Path

# Colab already has these. This only installs something if it is missing.
for _mod, _pkg in [("plotly", "plotly"), ("ipywidgets", "ipywidgets"), ("scipy", "scipy"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# Name of the Google Drive folder (inside My Drive) that holds the 4 data files.
DATA_FOLDER = "hackathon-barcelona-flood-model"

# ---- constants copied from ml/config.py (do not change them after seeing results) ----
STORM_DATES = ["2026-10-02", "2026-10-03"]
STORM_LABELS = {"2026-10-02": "2 Oct (light storm)", "2026-10-03": "3 Oct (heavy storm)"}
LABEL_DEPTH_M = 0.15          # flooded = simulated max depth of at least 0.15 m
THRESHOLD = 0.5               # probability above this is called "flood"
BLOCK_M = 500.0               # spatial blocks for cross-validation
CV_FOLDS = 5
STORM_HOURS = 3.0
TRAINED_DRAINAGE_MM_H = 20.0
RANDOM_SEED = 42
RADII_M = [150, 300]
TOP_N = 10
TOP_SPREAD_M = 200.0
STATIC_FEATURES = ["elevation_m", "slope_pct", "log_flow_acc", "twi", "depression_depth_m",
                   "dist_buried_stream_m", "dist_sea_m"]
STORM_FEATURES = ["excess_rain_mm"]
FEATURES = STATIC_FEATURES + STORM_FEATURES
HGB_PARAMS = dict(max_iter=200, learning_rate=0.05, max_depth=4, min_samples_leaf=40,
                  l2_regularization=1.0, early_stopping=False, random_state=42)
# Local map centre used by the project (ml/geo.py), to turn metres into map coordinates.
R_EARTH_M = 6371008.8
LAT0, LON0 = 41.385, 2.155

REQUIRED_FILES = ["dataset.csv", "test_points.csv", "metrics.json", "scenarios.json"]
print("Settings ready. Features used by the model:", FEATURES)

In [ ]:
# Load the 4 data files. In Colab: from Google Drive (with an upload fallback). Elsewhere: from the repo or FLOOD_DATA_DIR.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_data_folder():
    """Return a dict file name -> Path for the 4 files."""
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_dir = Path("/content/drive/MyDrive") / DATA_FOLDER
        work = Path("/content/flood_data")
        work.mkdir(exist_ok=True)
        missing = []
        for name in REQUIRED_FILES:
            src = drive_dir / name
            if src.exists():
                shutil.copy(src, work / name)
            else:
                missing.append(name)
        if missing:
            print("These files were not found in My Drive/" + DATA_FOLDER + ":", missing)
            print("INSTRUCTION: a file picker will open now. Click 'Choose files' and select these files from your computer:")
            for name in missing:
                print("   -", name)
            from google.colab import files
            uploaded = files.upload()
            for name, content in uploaded.items():
                (work / name).write_bytes(content)
            still = [n for n in REQUIRED_FILES if not (work / n).exists()]
            if still:
                raise FileNotFoundError(
                    "Still missing: " + ", ".join(still) + ". Put all 4 files in My Drive/" + DATA_FOLDER +
                    " (or upload them when asked) and run this cell again.")
        return {n: work / n for n in REQUIRED_FILES}
    env = os.environ.get("FLOOD_DATA_DIR")
    if env:
        folder = Path(env)
        found = {n: folder / n for n in REQUIRED_FILES}
        bad = [n for n, p in found.items() if not p.exists()]
        if bad:
            raise FileNotFoundError("FLOOD_DATA_DIR is set to " + str(folder) + " but these files are missing: " + ", ".join(bad))
        return found
    here = Path.cwd().resolve()
    for base in [here, *here.parents][:4]:
        paths = {"dataset.csv": base / "ml" / "out" / "dataset.csv",
                 "test_points.csv": base / "ml" / "out" / "test_points.csv",
                 "metrics.json": base / "data" / "ml" / "metrics.json",
                 "scenarios.json": base / "data" / "ml" / "scenarios.json"}
        if all(p.exists() for p in paths.values()):
            return paths
    raise FileNotFoundError("Could not find the data files from " + str(here) +
                            ". Run this notebook from ml/notebooks inside the repo, or set FLOOD_DATA_DIR to a folder holding the 4 files.")

DATA_PATHS = find_data_folder()
for name, path in DATA_PATHS.items():
    print(f"{name:16s} <- {path}")

df = pd.read_csv(DATA_PATHS["dataset.csv"])
df["excluded_edge_ponding"] = df["excluded_edge_ponding"].astype(bool)
df["name"] = df["name"].fillna("").replace("", "(unnamed street)")
test_points = pd.read_csv(DATA_PATHS["test_points.csv"])
with open(DATA_PATHS["metrics.json"], encoding="utf-8") as f:
    metrics = json.load(f)
with open(DATA_PATHS["scenarios.json"], encoding="utf-8") as f:
    scen = json.load(f)

print(f"\nLoaded dataset.csv: {len(df)} rows ({df['segment_id'].nunique()} street segments x {df['date'].nunique()} storms)")
print(f"Loaded test_points.csv: {len(test_points)} rows, columns: {list(test_points.columns)}")
print(f"Loaded scenarios.json: {len(scen['segments'])} segments, {len(scen['scenarios'])} what-if scenarios")

In [ ]:
# Trust level and the reasons, straight from metrics.json
trust = metrics["trust"]
print("Caveat:", scen["caveat"])
print()
print("TRUST LEVEL:", trust["level"].upper())
print("Why it is not higher:")
for reason in trust["reasons"]:
    print("  -", reason)
print()
rules = pd.DataFrame(trust["rules"])
rules["passed"] = rules["passed"].map({True: "yes", False: "NO"})
rules["value"] = rules["value"].map(lambda v: round(v, 3) if isinstance(v, float) else v)
display(rules.rename(columns={"rule": "Trust rule", "passed": "Passed?", "value": "Value"}))

### Behind the scenes: helper functions

The next cell holds the maths, copied from the project code (`ml/build_dataset.py`, `ml/evaluate.py`, `ml/geo.py`) so this notebook works on its own. You can skip reading it.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_curve
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, binom

_KX = R_EARTH_M * math.cos(math.radians(LAT0)) * math.pi / 180.0
_KY = R_EARTH_M * math.pi / 180.0

def to_xy(lon, lat):
    """lon/lat degrees -> local metres (x east, y north)."""
    return (np.asarray(lon, dtype=float) - LON0) * _KX, (np.asarray(lat, dtype=float) - LAT0) * _KY

def to_lonlat(x, y):
    return np.asarray(x, dtype=float) / _KX + LON0, np.asarray(y, dtype=float) / _KY + LAT0

def excess_rain(rain_mm, drainage_mm_h, storm_hours=STORM_HOURS):
    """Rain left after the drains have worked for storm_hours at their capacity."""
    return np.maximum(0.0, np.asarray(rain_mm, dtype=float) - drainage_mm_h * storm_hours)

def make_model():
    return HistGradientBoostingClassifier(**HGB_PARAMS)

def roc_auc(y, p):
    """Rank-based AUC. None when only one class is present."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    n_pos, n_neg = int(y.sum()), int((1 - y).sum())
    if n_pos == 0 or n_neg == 0:
        return None
    ranks = pd.Series(p).rank(method="average").to_numpy()
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def _safe_div(a, b):
    return float(a) / float(b) if b else None

def class_metrics(y, p, threshold=THRESHOLD):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    pred = (p >= threshold).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    n = tp + fp + fn + tn
    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    spec = _safe_div(tn, tn + fp)
    parts = [v for v in (recall, spec) if v is not None]
    majority = max(int(y.sum()), int((1 - y).sum()))
    edges = np.linspace(0, 1, 6)
    cal = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & ((p < hi) if hi < 1 else (p <= hi))
        if m.sum() > 0:
            cal.append({"bin": (round(float(lo), 2), round(float(hi), 2)), "mean_predicted": float(p[m].mean()),
                        "observed_rate": float(y[m].mean()), "n": int(m.sum())})
    return {"n": n, "confusion": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
            "accuracy": _safe_div(tp + tn, n), "baseline_accuracy": _safe_div(majority, n),
            "balanced_accuracy": float(np.mean(parts)) if parts else None,
            "precision": precision, "recall": recall, "roc_auc": roc_auc(y, p),
            "brier": float(np.mean((p - y) ** 2)) if n else None, "calibration": cal}

def training_frame(data):
    """Drop the edge-ponding rows (a closed-box artefact of the simulation), as the project does."""
    return data[~data["excluded_edge_ponding"]].reset_index(drop=True)

def spatial_cv(data, features=FEATURES, importance=True, n_splits=CV_FOLDS):
    """5-fold GroupKFold on 500 m blocks. Returns (frame with out-of-fold column 'oof', mean importance)."""
    d = training_frame(data)
    X, y, groups = d[features], d["label"].to_numpy(), d["block"].to_numpy()
    oof = np.full(len(d), np.nan)
    imps = {f: [] for f in features}
    for tr, te in GroupKFold(n_splits=n_splits).split(X, y, groups):
        model = make_model().fit(X.iloc[tr], y[tr])
        oof[te] = model.predict_proba(X.iloc[te])[:, 1]
        if importance and len(set(y[te])) == 2:
            r = permutation_importance(model, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=42)
            for f, v in zip(features, r.importances_mean):
                imps[f].append(float(v))
    mean_imp = {f: (float(np.mean(v)) if v else 0.0) for f, v in imps.items()}
    return d.assign(oof=oof), mean_imp

def leave_one_storm_out(data):
    d = training_frame(data)
    out = {}
    for tr_date, te_date in [("2026-10-02", "2026-10-03"), ("2026-10-03", "2026-10-02")]:
        tr, te = d[d["date"] == tr_date], d[d["date"] == te_date]
        model = make_model().fit(tr[FEATURES], tr["label"])
        p = model.predict_proba(te[FEATURES])[:, 1]
        out[(tr_date, te_date)] = class_metrics(te["label"].to_numpy(), p)
    return out

def greedy_top(p, x, y, n=TOP_N, spread_m=TOP_SPREAD_M):
    """Indices of the n highest-p points that are at least spread_m apart."""
    order = np.argsort(-np.asarray(p, dtype=float), kind="stable")
    chosen = []
    for i in order:
        if all(math.hypot(x[i] - x[j], y[i] - y[j]) >= spread_m for j in chosen):
            chosen.append(int(i))
            if len(chosen) == n:
                break
    return chosen

def hit_rate(points_xy, top_xy, radius_m):
    """Share of points within radius_m of any top area. Returns (hits, n, rate)."""
    if len(points_xy) == 0:
        return 0, 0, None
    d = np.hypot(points_xy[:, None, 0] - top_xy[None, :, 0], points_xy[:, None, 1] - top_xy[None, :, 1])
    hits = int((d.min(axis=1) <= radius_m).sum())
    return hits, len(points_xy), hits / len(points_xy)

# ---- map helpers: newer plotly has scatter_map, older has scatter_mapbox ----
HAS_NEW_MAP = hasattr(px, "scatter_map") and hasattr(go, "Scattermap")
px_scatter_map = px.scatter_map if HAS_NEW_MAP else px.scatter_mapbox
GoScatterMap = go.Scattermap if HAS_NEW_MAP else go.Scattermapbox
MAP_KEY = "map" if HAS_NEW_MAP else "mapbox"

def style_map(fig, lat, lon, zoom=12.5, height=620, title=None):
    fig.update_layout({MAP_KEY: dict(style="open-street-map", center=dict(lat=float(lat), lon=float(lon)), zoom=zoom)},
                      height=height, margin=dict(l=0, r=0, t=50 if title else 10, b=0),
                      title=title, legend=dict(orientation="h", yanchor="bottom", y=0.01, xanchor="left", x=0.01,
                                               bgcolor="rgba(255,255,255,0.8)"))
    return fig

# Colour-blind friendly colours used in the charts
C_DRY, C_FLOOD, C_GREY, C_DARK = "#4c78a8", "#e45756", "#8c8c8c", "#222222"
print("Helpers ready. Map style:", "scatter_map" if HAS_NEW_MAP else "scatter_mapbox (older plotly)")

## 2. The data: how many streets, how many flooded

Each row is one street segment in one storm. The "flooded" label is 1 when the simulated maximum water depth reached 0.15 m (about ankle height) or more. These labels are simulated, not observed.

A few segments on 3 Oct are marked "excluded". They sit next to the edge of the simulated box, where the simulation piles water against a wall. That is a quirk of the simulation, not a real flood, so the model does not train on them.

In [ ]:
summary = df.groupby("date").agg(
    rain_mm=("rain_mm", "first"),
    excess_rain_mm=("excess_rain_mm", "first"),
    street_segments=("segment_id", "size"),
    flooded_in_simulation=("label", "sum"),
    left_out_edge_ponding=("excluded_edge_ponding", "sum"),
)
summary["flooded_share"] = (summary["flooded_in_simulation"] / summary["street_segments"]).round(3)
train_part = training_frame(df).groupby("date")["label"].agg(rows_used_for_training="size", flooded_share_of_training_rows="mean").round(3)
summary = summary.join(train_part)
summary.index = [STORM_LABELS[d] for d in summary.index]
display(summary)

fig, ax = plt.subplots(figsize=(6, 3.2))
shares = summary["flooded_share_of_training_rows"]
ax.bar(shares.index, shares.values, color=[C_DRY, C_FLOOD])
ax.set_ylim(0, 1)
ax.set_ylabel("Share of streets flooded (simulated)")
ax.set_title("Flooded share per storm (training rows)")
for i, v in enumerate(shares.values):
    ax.text(i, v + 0.02, f"{v:.1%}", ha="center")
plt.tight_layout()
plt.show()
print("Very few streets flood on the light day, about half on the heavy day.")

## 3. Maps: where the simulation put water

One map per storm. Each dot is a street segment, coloured by the simulated maximum depth (pale = dry, dark blue = 1 m or more). Hover over a dot to see the street name, depth and label. Black dots are the rows left out of training because of the edge-ponding quirk.

In [ ]:
def depth_map(date):
    d = df[df["date"] == date].copy()
    keep = d[~d["excluded_edge_ponding"]]
    out = d[d["excluded_edge_ponding"]]
    fig = px_scatter_map(keep, lat="lat", lon="lon", color="depth_m", range_color=(0, 1.0),
                         color_continuous_scale="YlGnBu", hover_name="name",
                         hover_data={"depth_m": ":.2f", "label": True, "lat": False, "lon": False},
                         labels={"depth_m": "Simulated max depth (m)", "label": "Flooded (1) or dry (0)"})
    fig.update_traces(marker=dict(size=7))
    if len(out):
        fig.add_trace(GoScatterMap(lat=out["lat"], lon=out["lon"], mode="markers", marker=dict(size=9, color="black"),
                                   name="Left out of training (edge ponding)", text=out["name"] + ": " + out["depth_m"].round(2).astype(str) + " m",
                                   hoverinfo="text", showlegend=True))
    style_map(fig, d["lat"].mean(), d["lon"].mean(), title=f"{STORM_LABELS[date]}: simulated max depth ({d['rain_mm'].iloc[0]} mm rain)")
    fig.update_layout(coloraxis_colorbar=dict(title="Depth (m)"))
    return fig

for storm_date in STORM_DATES:
    depth_map(storm_date).show()

## 4. Feature explorer: what the model looks at

The model sees 7 facts about each street (terrain) plus one storm number. Below, each chart compares streets that flooded (red) with streets that stayed dry (blue), for each storm separately. The taller the bar, the more streets have that value.

**What to look for.** If red and blue look the same, that feature does not help tell flooded from dry streets. If red sits clearly to one side, the feature carries a signal. The small table after the charts gives one number per feature: 0.5 means no signal, near 1 means higher values go with flooding, near 0 means lower values go with flooding.

In [ ]:
train_df = training_frame(df)
fig, axes = plt.subplots(len(STATIC_FEATURES), 2, figsize=(11, 2.5 * len(STATIC_FEATURES)))
for i, feat in enumerate(STATIC_FEATURES):
    for j, date in enumerate(STORM_DATES):
        ax = axes[i, j]
        d = train_df[train_df["date"] == date]
        lo, hi = np.percentile(d[feat], [1, 99])
        bins = np.linspace(lo, hi, 30) if hi > lo else 20
        dry, fl = d.loc[d["label"] == 0, feat], d.loc[d["label"] == 1, feat]
        ax.hist(dry.clip(lo, hi), bins=bins, density=True, alpha=0.55, color=C_DRY, label=f"dry (n={len(dry)})")
        ax.hist(fl.clip(lo, hi), bins=bins, density=True, alpha=0.55, color=C_FLOOD, label=f"flooded (n={len(fl)})")
        ax.set_title(f"{feat} | {STORM_LABELS[date]}", fontsize=10)
        ax.set_yticks([])
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

single = pd.DataFrame({STORM_LABELS[date]: {feat: roc_auc(train_df.loc[train_df["date"] == date, "label"],
                                                          train_df.loc[train_df["date"] == date, feat])
                                            for feat in STATIC_FEATURES} for date in STORM_DATES}).round(2)
print("Signal of each single feature (0.5 = none). Read it per column, not across storms:")
display(single)
print("Note: the light storm has very few flooded streets, so its numbers are shaky.")
print("excess_rain_mm is not shown here: it has one value per storm, so it cannot split streets within a storm.")

## 5. Retrain and test fairly: spatial cross-validation

To test honestly, we cut the area into 500 m squares and split the squares into 5 groups. The model trains on 4 groups and is tested on the 5th, a part of town it has never seen. Every street gets a score from a model that did not train on its neighbourhood. This is called an out-of-fold score `p`.

The cell below retrains the model this way and then checks that we get the same numbers as the project's own `metrics.json`.

In [ ]:
oof_df, mean_imp = spatial_cv(df)
y_all, p_all = oof_df["label"].to_numpy(), oof_df["oof"].to_numpy()
cv = class_metrics(y_all, p_all)
within = {date: roc_auc(g["label"], g["oof"]) for date, g in oof_df.groupby("date")}
print(f"Trained and scored {len(oof_df)} rows with {CV_FOLDS}-fold spatial cross-validation.\n")

ref = metrics["metrics"]["spatial_cv"]
checks = [("accuracy", cv["accuracy"], ref["accuracy"]),
          ("balanced accuracy", cv["balanced_accuracy"], ref["balanced_accuracy"]),
          ("pooled AUC", cv["roc_auc"], ref["roc_auc"])]
print("REPRODUCTION CHECK against metrics.json (spatial_cv), allowed difference 0.01")
all_ok = True
for name, mine, theirs in checks:
    diff = mine - theirs
    ok = abs(diff) <= 0.01
    all_ok &= ok
    print(f"  {name:18s} notebook {mine:.4f} | metrics.json {theirs:.4f} | difference {diff:+.4f} | {'MATCH' if ok else 'DIFFERENT'}")
print("  confusion (tp, fp, fn, tn): notebook", tuple(cv["confusion"].values()), "| metrics.json", tuple(ref["confusion"].values()))
print("\nResult:", "reproduced within 0.01." if all_ok else "NOT reproduced within 0.01. The differences above are real. A different scikit-learn version or different data files are the usual cause.")
print(f"\nAUC inside each storm on its own: light storm {within['2026-10-02']:.2f}, heavy storm {within['2026-10-03']:.2f}")

### 5a. Confusion matrix and accuracy against a lazy baseline

The matrix counts the four ways a call can turn out:

- **True flood**: the model said flood, and the simulation flooded.
- **False alarm**: the model said flood, but the street stayed dry.
- **Missed flood**: the model said dry, but the street flooded.
- **True dry**: the model said dry, and it stayed dry.

On the right, accuracy is compared with a lazy model that always says "dry". If the model's accuracy is not clearly higher than the lazy one, accuracy alone is not telling us it is useful.

In [ ]:
def plot_confusion(ax, conf, title):
    """2x2 heatmap with plain-English names."""
    grid = np.array([[conf["tp"], conf["fn"]], [conf["fp"], conf["tn"]]])
    names = [["True flood", "Missed flood"], ["False alarm", "True dry"]]
    total = grid.sum()
    ax.imshow(grid / total, cmap="Blues", vmin=0, vmax=max(0.6, (grid / total).max()))
    for r in range(2):
        for c in range(2):
            share = grid[r, c] / total
            ax.text(c, r, f"{names[r][c]}\n{grid[r, c]}\n({share:.0%})", ha="center", va="center",
                    color="white" if share > 0.35 else "black", fontsize=11)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Model says flood", "Model says dry"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Simulation: flooded", "Simulation: dry"])
    ax.set_title(title, fontsize=11)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), gridspec_kw={"width_ratios": [1.2, 0.8, 1.0]})
plot_confusion(axes[0], cv["confusion"], "Spatial cross-validation (all rows)")

base_dry = float((y_all == 0).mean())
axes[1].bar(["Always says\n'dry'", "The model"], [base_dry, cv["accuracy"]], color=[C_GREY, C_DRY])
axes[1].set_ylim(0, 1)
axes[1].set_title("Accuracy: model vs lazy baseline")
for i, v in enumerate([base_dry, cv["accuracy"]]):
    axes[1].text(i, v + 0.02, f"{v:.1%}", ha="center")

labels = ["Balanced\naccuracy", "Precision", "Recall"]
vals = [cv["balanced_accuracy"], cv["precision"], cv["recall"]]
axes[2].bar(labels, vals, color=C_DRY)
axes[2].set_ylim(0, 1)
axes[2].set_title("Other scores")
for i, v in enumerate(vals):
    axes[2].text(i, v + 0.02, f"{v:.2f}", ha="center")
plt.tight_layout()
plt.show()

print(f"Accuracy {cv['accuracy']:.1%} versus {base_dry:.1%} for always saying dry. The model is {'no better than' if cv['accuracy'] <= base_dry + 0.005 else 'slightly better than'} the lazy baseline on accuracy.")
print(f"Balanced accuracy {cv['balanced_accuracy']:.2f}: the average of how many flooded streets it catches (recall {cv['recall']:.2f}) and how many dry streets it leaves alone.")
print(f"Precision {cv['precision']:.2f}: when it says flood, it is right about {cv['precision']:.0%} of the time.")

### 5b. ROC curves: how well does it rank streets?

An ROC curve shows how well the scores put flooded streets above dry ones. The dashed diagonal is a coin flip. Further towards the top-left corner is better.

Read the blue line carefully. It pools both storms, so it gets a boost just from telling the heavy day from the light day. The line for each storm on its own shows how well the model ranks streets *within* one storm, which is the more honest test of the terrain.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.6))
fpr, tpr, _ = roc_curve(y_all, p_all)
ax.plot(fpr, tpr, color=C_DRY, lw=2.5, label=f"Both storms pooled (AUC {cv['roc_auc']:.2f})")
for date, color in zip(STORM_DATES, ["#59a14f", C_FLOOD]):
    g = oof_df[oof_df["date"] == date]
    fpr_s, tpr_s, _ = roc_curve(g["label"], g["oof"])
    ax.plot(fpr_s, tpr_s, color=color, lw=1.8, label=f"{STORM_LABELS[date]} alone (AUC {within[date]:.2f})")
ax.plot([0, 1], [0, 1], "--", color=C_GREY, label="Coin flip (AUC 0.50)")
ax.set_xlabel("False alarm rate (dry streets called flood)")
ax.set_ylabel("Hit rate (flooded streets caught)")
ax.set_title("ROC curves, spatial cross-validation")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()
print(f"Pooled AUC {cv['roc_auc']:.2f} looks decent, but inside the heavy storm alone it is {within['2026-10-03']:.2f}.")
print("That means most of the pooled score comes from knowing which day it is, not from knowing which streets are low-lying.")

### 5c. Calibration: can you read `p` as a percentage?

We sort streets into 5 groups by their score `p` (0 to 0.2, 0.2 to 0.4, and so on). For each group we compare the average score (x axis) with the share that really flooded in the simulation (y axis). If `p` were a true percentage, the dots would sit on the dashed diagonal. Dot size shows how many streets are in the group.

In [ ]:
cal = pd.DataFrame(cv["calibration"])
fig, ax = plt.subplots(figsize=(5.8, 5.4))
ax.plot([0, 1], [0, 1], "--", color=C_GREY, label="Perfect calibration")
ax.plot(cal["mean_predicted"], cal["observed_rate"], "-", color=C_DRY, alpha=0.6)
ax.scatter(cal["mean_predicted"], cal["observed_rate"], s=40 + cal["n"] / 6, color=C_DRY, zorder=3, label="Score groups (size = number of streets)")
for _, r in cal.iterrows():
    ax.annotate(f"n={int(r['n'])}", (r["mean_predicted"], r["observed_rate"]), textcoords="offset points", xytext=(8, -12), fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Average score p in the group")
ax.set_ylabel("Share that really flooded (simulated)")
ax.set_title("Calibration (5 bins)")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()
print("Compare each dot with the diagonal. Dots far from the diagonal mean p is not a reliable percentage.")
top_bin = cal.iloc[-1]
print(f"In the top group (scores {top_bin['bin'][0]:.1f} to {top_bin['bin'][1]:.1f}) the average score is {top_bin['mean_predicted']:.2f}, and {top_bin['observed_rate']:.0%} of those streets flooded in the simulation.")
print("Safest reading: p is 'higher than other streets', not a percentage chance.")

### 5d. Which inputs matter? Permutation importance

To measure how much the model relies on an input, we scramble that input in the test part and see how much the ranking score (AUC) drops. A big drop means the model leans on it. The bars are scaled to add up to 100%.

In [ ]:
clipped = {k: max(0.0, v) for k, v in mean_imp.items()}
total = sum(clipped.values())
imp = pd.Series({k: (v / total if total > 0 else 0.0) for k, v in clipped.items()}).sort_values()
ref_imp = pd.Series({r["feature"]: r["importance"] for r in metrics["model"]["feature_importance"]})

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(imp.index, imp.values, color=[C_FLOOD if k == "excess_rain_mm" else C_DRY for k in imp.index])
for i, v in enumerate(imp.values):
    ax.text(v + 0.01, i, f"{v:.1%}", va="center", fontsize=9)
ax.set_xlim(0, 1.1)
ax.set_xlabel("Share of total importance")
ax.set_title("Permutation feature importance (spatial cross-validation)")
plt.tight_layout()
plt.show()

compare = pd.DataFrame({"this notebook": imp, "metrics.json": ref_imp}).sort_values("metrics.json", ascending=False).round(3)
display(compare)
print(f"The storm number (excess_rain_mm) carries {imp['excess_rain_mm']:.0%} of the importance. The model mostly learned 'heavy day versus light day'.")

## 6. Leave one storm out: can it carry over to a different storm?

A harder test: train on one storm only, then test on the other. If the model had really learned how water behaves, it should cope with a storm it never saw. With only two storms, one very light and one very heavy, this is a tough ask, and the results show it.

In [ ]:
loso = leave_one_storm_out(df)
ref_loso = {(a, b): metrics["metrics"]["leave_one_storm_out"][f"train_{a.replace('-', '_')}_test_{b.replace('-', '_')}"] for (a, b) in loso}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, ((tr_date, te_date), m) in zip(axes, loso.items()):
    plot_confusion(ax, m["confusion"], f"Train on {STORM_LABELS[tr_date]}\ntest on {STORM_LABELS[te_date]}  (AUC {m['roc_auc']:.2f})")
plt.tight_layout()
plt.show()

rows = []
for (tr_date, te_date), m in loso.items():
    r = ref_loso[(tr_date, te_date)]
    rows.append({"Train on": STORM_LABELS[tr_date], "Test on": STORM_LABELS[te_date],
                 "AUC": round(m["roc_auc"], 3), "AUC in metrics.json": round(r["roc_auc"], 3),
                 "Accuracy": round(m["accuracy"], 3), "Most-common-answer baseline": round(m["baseline_accuracy"], 3),
                 "Precision": round(m["precision"], 3), "Recall": round(m["recall"], 3)})
display(pd.DataFrame(rows))
a = loso[("2026-10-02", "2026-10-03")]; b = loso[("2026-10-03", "2026-10-02")]
print(f"Train light, test heavy: AUC {a['roc_auc']:.2f} (a coin flip is 0.50). It catches only {a['recall']:.0%} of the flooded streets.")
print(f"Train heavy, test light: AUC {b['roc_auc']:.2f}. It flags a lot of streets that stay dry (precision {b['precision']:.2f}).")

## 7. Scenario explorer: pick a what-if

Choose an amount of rain and a drainage strength. The map shows the model's score `p` for every street (pale yellow = low, dark red = high). The 10 highest-scoring areas, kept at least 200 m apart, are the big numbered dots.

These scenarios were scored with the final model that is saved in `scenarios.json`. Remember that `p` is a ranking score, not a chance of real flooding. A scenario is flagged **extrapolated** when it lies outside what the model was trained on.

If the dropdowns do not appear or the map stays empty, use the plain function call in the cell after. It does the same thing.

In [ ]:
SEGMENTS = pd.DataFrame(scen["segments"])
SEGMENTS["name"] = SEGMENTS["name"].replace("", "(unnamed street)")
SC_BY_ID = {s["id"]: s for s in scen["scenarios"]}
TRAINED_RAIN_LO, TRAINED_RAIN_HI = metrics["model"]["trained_rain_mm"]

def scenario_notes(rain_mm, drainage_mm_h):
    """Plain-English reasons why a scenario is, or is not, extrapolated."""
    notes = []
    if rain_mm < TRAINED_RAIN_LO or rain_mm > TRAINED_RAIN_HI:
        notes.append(f"rain {rain_mm} mm is outside the {TRAINED_RAIN_LO} to {TRAINED_RAIN_HI} mm the model was trained on")
    if float(drainage_mm_h) != TRAINED_DRAINAGE_MM_H:
        notes.append(f"drainage {drainage_mm_h} mm/h was never simulated, only {TRAINED_DRAINAGE_MM_H:.0f} mm/h was")
    return notes

def show_scenario(rain_mm, drainage_mm_h):
    """Draw the map for one what-if scenario and print its summary."""
    sid = f"r{int(rain_mm)}_d{int(drainage_mm_h)}"
    if sid not in SC_BY_ID:
        print("No such scenario. Available:", sorted(SC_BY_ID))
        return
    sc = SC_BY_ID[sid]
    seg = SEGMENTS.assign(p=sc["p"])
    top = pd.DataFrame(sc["top"])
    top["label_name"] = top["name"].replace("", "(unnamed street)")

    fig = px_scatter_map(seg, lat="lat", lon="lon", color="p", range_color=(0, 1), color_continuous_scale="YlOrRd",
                         hover_name="name", hover_data={"p": ":.2f", "elev_m": True, "lat": False, "lon": False},
                         labels={"p": "Score p", "elev_m": "Elevation (m)"})
    fig.update_traces(marker=dict(size=7))
    fig.add_trace(GoScatterMap(lat=top["lat"], lon=top["lon"], mode="markers+text", text=top["rank"].astype(str),
                               textfont=dict(color="white", size=11), marker=dict(size=22, color=C_DARK),
                               customdata=np.column_stack([top["label_name"], top["p"].round(2)]),
                               hovertemplate="Rank %{text}: %{customdata[0]}<br>p = %{customdata[1]}<extra></extra>",
                               name="Top 10 areas"))
    style_map(fig, seg["lat"].mean(), seg["lon"].mean(),
              title=f"Rain {rain_mm} mm, drainage {drainage_mm_h} mm/h (what-if scenario)")
    fig.update_layout(coloraxis_colorbar=dict(title="p"))
    fig.show()

    notes = scenario_notes(rain_mm, drainage_mm_h)
    print(f"Scenario: rain {rain_mm} mm, drainage {drainage_mm_h} mm/h  (excess rain the model sees: {float(excess_rain(rain_mm, drainage_mm_h)):.1f} mm)")
    print(f"  risk_score    : {sc['risk_score']}  (0 to 10, average score over all streets)")
    print(f"  share_flooded : {sc['share_flooded']:.1%}  (streets with p of 0.5 or more)")
    print(f"  extrapolated  : {'YES' if sc['extrapolated'] else 'no'}")
    if sc["extrapolated"]:
        for n in notes:
            print("     reason:", n)
    else:
        print(f"     rain is inside {TRAINED_RAIN_LO} to {TRAINED_RAIN_HI} mm and drainage equals {TRAINED_DRAINAGE_MM_H:.0f} mm/h")
        print("     but the model has seen only two storm sizes, so in-between values behave like the nearest one")
    print("  Top 10 areas:")
    for _, r in top.iterrows():
        print(f"    {int(r['rank']):2d}. {r['label_name']}  (p = {r['p']:.2f})")

rain_options = sorted({s["rain_mm"] for s in scen["scenarios"]})
drain_options = sorted({s["drainage_mm_h"] for s in scen["scenarios"]})
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    dd_rain = widgets.Dropdown(options=rain_options, value=rain_options[-1], description="Rain (mm):")
    dd_drain = widgets.Dropdown(options=drain_options, value=20 if 20 in drain_options else drain_options[0],
                                description="Drainage (mm/h):", style={"description_width": "initial"})
    out = widgets.Output()
    def _redraw(_change=None):
        with out:
            clear_output(wait=True)
            show_scenario(dd_rain.value, dd_drain.value)
    dd_rain.observe(_redraw, names="value")
    dd_drain.observe(_redraw, names="value")
    display(widgets.HBox([dd_rain, dd_drain]), out)
    _redraw()
except Exception as err:
    print("Dropdowns could not be shown here (" + type(err).__name__ + "). Use the plain function call in the next cell.")

In [ ]:
# Same thing without widgets. Change the two numbers and run the cell.
# Allowed rain: 25, 50, 75, 100, 143. Allowed drainage: 10, 20, 30.
show_scenario(rain_mm=75, drainage_mm_h=10)

## 8. Real-world check: do the top areas match places that are known to flood?

Here is a sanity check against reality. We take the model's top 10 areas for the 143 mm rain, 20 mm/h drainage scenario and compare them with **reported flood places** from the project's test list (city flood-study hotspots, sewer-stress zones and one report from October 2026). We count how many reported places sit within 150 m and 300 m of a top area, and compare with 3,000 random points in the same box.

The circles on the map show the 150 m (inner) and 300 m (outer) reach around each top area.

Be careful: there are only 10 reported places, so this check cannot prove anything either way.

In [ ]:
# Segment positions in metres (same order as the segments in scenarios.json)
seg_static = df[df["date"] == df["date"].iloc[0]].reset_index(drop=True)
assert (seg_static["segment_id"].astype(str).to_numpy() == np.array([s["id"] for s in scen["segments"]])).all(), "segment order differs"
sc_main = SC_BY_ID["r143_d20"]
top_idx = [t["segment_index"] for t in sc_main["top"]]
top_xy = np.column_stack([seg_static["x_m"].to_numpy()[top_idx], seg_static["y_m"].to_numpy()[top_idx]])
top_ll = pd.DataFrame(sc_main["top"])

reported = test_points[test_points["kind"] == "reported"].reset_index(drop=True)
randoms = test_points[test_points["kind"] == "random"]
rep_xy, rnd_xy = reported[["x_m", "y_m"]].to_numpy(), randoms[["x_m", "y_m"]].to_numpy()

def circle_lines(radius_m, n=60):
    lons, lats = [], []
    ang = np.linspace(0, 2 * np.pi, n + 1)
    for x0, y0 in top_xy:
        lo, la = to_lonlat(x0 + radius_m * np.cos(ang), y0 + radius_m * np.sin(ang))
        lons += list(lo) + [None]
        lats += list(la) + [None]
    return lons, lats

fig = go.Figure()
for radius, color in zip(RADII_M, ["#222222", "#8c8c8c"]):
    lons, lats = circle_lines(radius)
    fig.add_trace(GoScatterMap(lon=lons, lat=lats, mode="lines", line=dict(width=2, color=color), name=f"{radius} m circle", hoverinfo="skip"))
fig.add_trace(GoScatterMap(lat=top_ll["lat"], lon=top_ll["lon"], mode="markers+text", text=top_ll["rank"].astype(str),
                           textfont=dict(color="white", size=11), marker=dict(size=22, color=C_DARK),
                           customdata=top_ll["name"].replace("", "(unnamed street)"),
                           hovertemplate="Top area %{text}: %{customdata}<extra></extra>", name="Model top 10 areas"))
layer_colors = {"resccue": "#e45756", "sewer_t10": "#f58518", "proof_oct2026": "#59a14f"}
layer_names = {"resccue": "Reported: city flood-study hotspot", "sewer_t10": "Reported: sewer-stress zone", "proof_oct2026": "Reported: Oct 2026 incident"}
for layer, g in reported.groupby("layer"):
    fig.add_trace(GoScatterMap(lat=g["lat"], lon=g["lon"], mode="markers", marker=dict(size=13, color=layer_colors.get(layer, "#b279a2")),
                               text=g["name"], hovertemplate="%{text}<extra></extra>", name=layer_names.get(layer, layer)))
style_map(fig, seg_static["lat"].mean(), seg_static["lon"].mean(), title="Reported flood places versus model top 10 areas (143 mm rain, 20 mm/h drainage)")
fig.show()

ref_rw = {r["radius_m"]: r for r in metrics["metrics"]["real_world"]}
print(f"Reported places: {len(reported)}.  Random comparison points: {len(randoms)}.\n")
for radius in RADII_M:
    hits, n, rate = hit_rate(rep_xy, top_xy, radius)
    _, _, rrate = hit_rate(rnd_xy, top_xy, radius)
    lift = rate / rrate if rrate else float("nan")
    luck = float(binom.sf(hits - 1, n, rrate)) if hits > 0 else 1.0
    r = ref_rw.get(radius, {})
    print(f"Within {radius} m: {hits} of {n} reported places hit ({rate:.0%}). Random points hit {rrate:.1%}. Lift {lift:.2f}x.")
    if hits > 0:
        print(f"    Chance of getting {hits} or more hits out of {n} by luck alone, if the top areas were placed at random: {luck:.0%}")
    if r:
        print(f"    metrics.json says: {r['hits']} of {r['n_points']}, random {r['random_rate']:.1%}, lift {r['lift']:.2f}x")
same = all(hit_rate(rep_xy, top_xy, r)[0] == ref_rw[r]["hits"] for r in RADII_M if r in ref_rw)
print("\nHit counts match metrics.json." if same else "\nHit counts differ from metrics.json. This scenario list uses 143 mm and leaves out edge-ponding segments, while the project check used 143.4 mm.")

dist_to_top = np.hypot(rep_xy[:, None, 0] - top_xy[None, :, 0], rep_xy[:, None, 1] - top_xy[None, :, 1]).min(axis=1)
table = reported[["name", "layer"]].assign(nearest_top_area_m=dist_to_top.round(0).astype(int))
table["within_150m"] = np.where(dist_to_top <= 150, "yes", "no")
table["within_300m"] = np.where(dist_to_top <= 300, "yes", "no")
display(table.sort_values("nearest_top_area_m").reset_index(drop=True))
print("\nHonest reading: with only 10 reported places, a small hit rate or a small lift is well within luck. It is not evidence the model works on real floods.")

## 9. Why terrain alone is at chance

Inside the heavy storm, ranking streets by terrain is no better than a coin flip. Is that a bug, or is something real going on? Three checks:

1. **Neighbour check.** Do nearby streets get similar simulated depths? If yes, the labels hold a real street pattern and are not random noise.
2. **Single-feature check.** Does any one terrain feature line up with depth inside a storm? We use Spearman correlation (+1 and -1 are a perfect match, 0 is none).
3. **Day-to-day check.** Do the same streets flood on both days?

In [ ]:
# 1. Neighbour check: average depth of the 5 nearest segments versus the segment's own depth
rng = np.random.default_rng(RANDOM_SEED)
nb_rows = []
for date in STORM_DATES:
    d = train_df[train_df["date"] == date]      # edge-ponding rows left out, as in training
    xy = d[["x_m", "y_m"]].to_numpy()
    depth = d["depth_m"].to_numpy()
    _, idx = cKDTree(xy).query(xy, k=6)          # column 0 is the segment itself
    idx = idx[:, 1:]
    rho_real = spearmanr(depth, depth[idx].mean(axis=1))[0]
    shuffled = rng.permutation(depth)
    rho_shuf = spearmanr(shuffled, shuffled[idx].mean(axis=1))[0]
    nb_rows.append({"Storm": STORM_LABELS[date], "Neighbour correlation": round(rho_real, 2), "Same, depths shuffled": round(rho_shuf, 2)})
print("1. Does a street's depth match the average depth of its 5 nearest neighbours?")
display(pd.DataFrame(nb_rows))
print("   Real values clearly above zero, with shuffled ones near zero, mean the simulated labels contain a genuine spatial pattern.\n")

# 2. Spearman of each terrain feature with depth, within each storm
sp = pd.DataFrame({STORM_LABELS[date]: {feat: spearmanr(train_df.loc[train_df["date"] == date, feat],
                                                         train_df.loc[train_df["date"] == date, "depth_m"])[0]
                                        for feat in STATIC_FEATURES} for date in STORM_DATES})
print("2. Spearman correlation of each terrain feature with simulated depth, inside each storm")
display(sp.round(2))
ax = sp.plot.barh(figsize=(7.5, 4), color=[C_DRY, C_FLOOD])
ax.axvline(0, color="black", lw=0.8)
ax.axvspan(-0.2, 0.2, color=C_GREY, alpha=0.15)
ax.set_xlim(-1, 1)
ax.set_xlabel("Spearman correlation with depth (grey band: between -0.2 and 0.2)")
ax.set_title("How closely does each terrain feature track depth?")
plt.tight_layout()
plt.show()
print(f"   Largest absolute correlation: {sp.abs().to_numpy().max():.2f}\n")

# 3. Do the two days agree on which streets flooded?
wide = df.pivot(index="segment_id", columns="date", values="label")
agree_all = float((wide["2026-10-02"] == wide["2026-10-03"]).mean())
ok_ids = df.groupby("segment_id")["excluded_edge_ponding"].any()
ok_ids = ok_ids[~ok_ids].index
wide_ok = wide.loc[wide.index.isin(ok_ids)]
agree_ok = float((wide_ok["2026-10-02"] == wide_ok["2026-10-03"]).mean())
print("3. Do the two days agree on the label for each street?")
display(pd.crosstab(wide_ok["2026-10-02"].map({0: "dry on 2 Oct", 1: "flooded on 2 Oct"}),
                    wide_ok["2026-10-03"].map({0: "dry on 3 Oct", 1: "flooded on 3 Oct"})))
print(f"   Same label on both days: {agree_all:.0%} of all segments ({agree_ok:.0%} excluding edge-ponding segments).")
print("   Most streets are dry on 2 Oct, so the heavy day floods about half of everything and the days disagree for the rest.\n")
print("Takeaway: the simulated labels are not noise, but the terrain facts used here do not explain which streets the simulation floods.")
print("The simulation may depend on things this model cannot see, such as how the drains and building surfaces work. This is a guess, not tested.")

## 10. Sandbox: test your own improvement idea

Got an idea for a new input (a "feature")? Write a small function that adds one or more columns to the table, and the cell re-runs the fair spatial test. It reports the AUC inside each storm, which is the number to watch.

**The recipe**

1. Write `my_idea(d)` that takes the table `d`, adds new column(s), and returns `d` and the list of new column names.
2. Run `evaluate_idea("my idea", my_idea)`.
3. Compare the AUC with the baseline. Differences smaller than about 0.02 are probably luck, which is why a pure-noise column is included as a reality check.

Never use the `label`, `depth_m` or `flood_class` columns to build a feature. That would be cheating: the model would see the answer.

In [ ]:
def evaluate_idea(name, add_features):
    """Add feature columns with add_features(d) -> (d, [new column names]), rerun spatial CV, compare AUC with the baseline."""
    d, new_cols = add_features(df.copy())
    assert not ({"label", "depth_m", "flood_class"} & set(new_cols)), "Do not use the answer columns as features."
    res, _ = spatial_cv(d, features=FEATURES + list(new_cols), importance=False)
    rows = []
    for tag, frame in [("Baseline (current 8 features)", oof_df), (f"With: {name}", res)]:
        w = {date: roc_auc(g["label"], g["oof"]) for date, g in frame.groupby("date")}
        rows.append({"Model": tag, "AUC light storm": w["2026-10-02"], "AUC heavy storm": w["2026-10-03"],
                     "AUC pooled": roc_auc(frame["label"], frame["oof"])})
    out = pd.DataFrame(rows).set_index("Model")
    out.loc["Change"] = out.iloc[1] - out.iloc[0]
    display(out.round(3))
    change = out.loc["Change", "AUC heavy storm"]
    print(f"Heavy-storm AUC change: {change:+.3f}.", "Probably within noise (under 0.02)." if abs(change) < 0.02 else "Large enough to look at more closely.")
    return res

# --- Worked toy example: "how low is this street compared with the others in its 500 m block?" ---
def elevation_rank_in_block(d):
    # 0 = lowest street in its block, 1 = highest. Uses only terrain, never the answer.
    d["elev_rank_in_block"] = d.groupby(["date", "block"])["elevation_m"].rank(pct=True)
    return d, ["elev_rank_in_block"]

_ = evaluate_idea("elevation rank within block", elevation_rank_in_block)

# --- Reality check: a column of pure random noise. Any gain it shows is luck. ---
def random_noise_feature(d):
    d["noise"] = np.random.default_rng(0).random(len(d))
    return d, ["noise"]

_ = evaluate_idea("pure random noise (should not help)", random_noise_feature)

In [ ]:
# YOUR TURN. Edit the body of my_idea, then run this cell.
# The table d has one row per street and storm. Useful columns: x_m, y_m (metres), block, date, and the 8 features.
def my_idea(d):
    # Example: how much higher or lower is this street than the average in its block?
    d["elev_minus_block_mean"] = d["elevation_m"] - d.groupby(["date", "block"])["elevation_m"].transform("mean")
    return d, ["elev_minus_block_mean"]

_ = evaluate_idea("my idea", my_idea)

## 11. Summary: the honest numbers, and what would improve it

The table puts this notebook's numbers next to the project's `metrics.json`. After it comes the list of improvements from `metrics.json`, in the order the project proposes them.

In [ ]:
r300 = next(r for r in metrics["metrics"]["real_world"] if r["radius_m"] == 300)
summ = pd.DataFrame([
    ("Spatial CV accuracy (always-dry baseline in brackets)", f"{cv['accuracy']:.3f} ({base_dry:.3f})", f"{ref['accuracy']:.3f} ({ref['baseline_accuracy']:.3f})"),
    ("Spatial CV balanced accuracy", f"{cv['balanced_accuracy']:.3f}", f"{ref['balanced_accuracy']:.3f}"),
    ("Spatial CV pooled AUC (both storms mixed)", f"{cv['roc_auc']:.3f}", f"{ref['roc_auc']:.3f}"),
    ("AUC inside the light storm only", f"{within['2026-10-02']:.3f}", "about 0.84 (project README)"),
    ("AUC inside the heavy storm only", f"{within['2026-10-03']:.3f}", "about 0.51 (project README)"),
    ("Train light, test heavy: AUC", f"{a['roc_auc']:.3f}", f"{ref_loso[('2026-10-02', '2026-10-03')]['roc_auc']:.3f}"),
    ("Train heavy, test light: AUC", f"{b['roc_auc']:.3f}", f"{ref_loso[('2026-10-03', '2026-10-02')]['roc_auc']:.3f}"),
    ("Real-world hits within 300 m (top 10 areas)", f"{hit_rate(rep_xy, top_xy, 300)[0]} of {len(rep_xy)}", f"{r300['hits']} of {r300['n_points']}"),
    ("Trust level", "see section 1", trust["level"]),
], columns=["Check", "This notebook", "metrics.json"])
display(summ)

print("\nIN PLAIN ENGLISH")
print(f"- Accuracy ({cv['accuracy']:.1%}) is no better than always guessing dry ({base_dry:.1%}).")
print(f"- The pooled AUC ({cv['roc_auc']:.2f}) looks good, but mostly because the model can tell the heavy day from the light day.")
print(f"- Inside the heavy storm, ranking streets by terrain is close to a coin flip (AUC {within['2026-10-03']:.2f}).")
print("- Trained on one storm, it does not carry over to the other.")
print("- The real-world check rests on 10 places, too few to count as proof.")
print("- All labels come from our own physics simulation, not from real floods.")
print("\nWHAT WOULD IMPROVE IT (from metrics.json)")
for i, item in enumerate(metrics["improvements"], 1):
    print(f"{i:2d}. {item}")

### Where to go next

The biggest gains are in the data, not in the model: more simulated storms of different sizes with open box edges, real incident reports as labels, and the physics simulation's own input layers as features. Use the sandbox in section 10 to try cheap ideas first, but expect little change until the data grows.